# AI Cycling Coach — GPU Training (Kaggle)

**Settings (right sidebar) → Accelerator → GPU T4 x2** before running.

Then click **Run All**. No uploads needed — generates data here (~3 min) then trains on GPU (~1–2 h).

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── 2. Clone repo + set paths ────────────────────────────────────────────────
import os, sys

REPO    = 'https://github.com/yossibello/ai-coach.git'
WORKDIR = '/kaggle/working/ai-coach'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    !cd {WORKDIR} && git pull

%cd {WORKDIR}

for p in [f'{WORKDIR}/backend', WORKDIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH']      = f'{WORKDIR}/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models
print('cwd:', os.getcwd())
print('sys.path[0:3]:', sys.path[:3])

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install pyarrow --upgrade -q
import torch, pandas
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('pandas:', pandas.__version__)

In [ ]:

# ── 4. Load training data ────────────────────────────────────────────────────
# TWO OPTIONS — set MODE below:
#
#   'dataset'  → you added the ai-coach-synthetic Kaggle dataset (recommended, instant)
#   'generate' → generate here in Kaggle (~3 min for 20K, ~8 min for 50K)

import os, sys, subprocess, multiprocessing, pandas as pd

MODE          = 'generate'  # ← 'dataset' | 'generate'
ATHLETES      = 50_000      # only used when MODE='generate'
DATASET_PATH  = '/kaggle/input/ai-coach-synthetic/synthetic.parquet'
DATA_FILE     = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if MODE == 'dataset':
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_PATH}\n'
            'Add it: right panel → Add data → search "ai-coach-synthetic"'
        )
    print(f'Using Kaggle dataset: {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)')
    DATA_FILE = DATASET_PATH

elif MODE == 'generate':
    workers = max(1, multiprocessing.cpu_count() - 1)
    print(f'Generating {ATHLETES:,} athletes using {workers} workers…')

    # Use subprocess so we capture stdout+stderr and get a real exception on failure.
    # The !shell magic swallows errors and leaves DATA_FILE missing.
    env = os.environ.copy()
    env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"
    result = subprocess.run(
        [sys.executable, '-m', 'ml.training.generate_synthetic',
         '--athletes', str(ATHLETES),
         '--workers',  str(workers),
         '--output',   DATA_FILE],
        env=env,
        capture_output=False,   # stream output live to the cell
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'generate_synthetic failed (exit {result.returncode}).\n'
            'Scroll up for the traceback printed above.'
        )

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Expected {DATA_FILE} but it was not created.\n'
        'Check the output above for errors.'
    )

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'pc_5s_wkg'       in df.columns, 'Old parquet — missing power curve features, regenerate'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')
print(f'  Power curve columns present: pc_5s_wkg={df.pc_5s_wkg.notna().mean()*100:.0f}% non-null')
del df


In [ ]:

# ── 5. Train ─────────────────────────────────────────────────────────────────
import sys, os, torch, argparse, multiprocessing

n_gpus  = torch.cuda.device_count() if torch.cuda.is_available() else 1
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

if   vram_gb >= 45: per_gpu = 4096
elif vram_gb >= 38: per_gpu = 2048
elif vram_gb >= 20: per_gpu = 1024
else:               per_gpu = 1024

BATCH_SIZE      = per_gpu * n_gpus
STEPS_PER_EPOCH = 5000
EPOCHS          = 50
MODEL_FILE      = 'backend/models/cycling_coach.pt'
LAST_FILE       = 'backend/models/cycling_coach_last.pt'
dl_workers      = min(6, multiprocessing.cpu_count())
os.environ['DATALOADER_WORKERS'] = str(dl_workers)

LEARNING_RATE = 3e-4

# ── Checkpoint / resume logic ─────────────────────────────────────────────────
# Prefer _last.pt (actual last epoch) over best .pt (best val epoch).
# Set FORCE_FRESH=True to ignore all checkpoints and start from epoch 1.
FORCE_FRESH = False

if FORCE_FRESH:
    for _f in [MODEL_FILE, LAST_FILE]:
        if os.path.exists(_f): os.remove(_f)
    print('*** FORCE_FRESH: deleted checkpoints, starting from epoch 1 ***')
    CHECKPOINT = None
else:
    # _last.pt resumes from actual last epoch; fall back to best .pt
    if os.path.exists(LAST_FILE):
        CHECKPOINT = LAST_FILE
    elif os.path.exists(MODEL_FILE):
        CHECKPOINT = MODEL_FILE
    else:
        CHECKPOINT = None

    if CHECKPOINT:
        _meta = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
        _ep   = _meta.get('metrics', {}).get('epoch', '?') if isinstance(_meta, dict) else '?'
        print(f'*** Resuming from {os.path.basename(CHECKPOINT)}: epoch {_ep} → will train epochs {_ep+1 if isinstance(_ep,int) else "?"}-{EPOCHS} ***')
    else:
        print('*** No checkpoint found — starting fresh ***')

print(f'GPUs: {n_gpus}  |  VRAM/GPU: {vram_gb:.1f} GB  |  batch: {BATCH_SIZE} ({per_gpu}/GPU)')
print(f'Steps/epoch: {STEPS_PER_EPOCH}  |  LR: {LEARNING_RATE}  |  Workers: {dl_workers}')
print(f'Data: {DATA_FILE}  ({os.path.getsize(DATA_FILE)/1e6:.0f} MB)')
print('─' * 60)

for _k in list(sys.modules.keys()):
    if 'training.train' in _k:
        del sys.modules[_k]

from ml.training.train import train as run_training
os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = CHECKPOINT,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = LEARNING_RATE,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,
    nhead            = 8,
    num_layers       = 8,
    d_ff             = 1024,
    dropout          = 0.15,
    fast             = False,
    patience         = 20,
    compile          = False,
    no_amp           = False,
    mask_workout_type = False,
    # Outcome-weighting flags (ignored for pure-synthetic pretrain — all
    # weights are 1.0 here; they only bite on source=real data).
    no_outcome_weighting = False,
    outcome_scale        = 0.05,
    outcome_deadband     = 0.0,
)

run_training(args)
print(f'\n✓ Training complete! Model → {MODEL_FILE}')


In [ ]:
# ── 6. Copy models to /kaggle/working/ so Kaggle saves them as output ─────────
import shutil, os

for src, dst in [
    (MODEL_FILE,                          '/kaggle/working/cycling_coach.pt'),
    ('backend/models/cycling_coach_last.pt', '/kaggle/working/cycling_coach_last.pt'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✓ {os.path.basename(src)} → {dst}')

print('  → After the notebook finishes, go to the Output tab and download both.')
print('  → Use cycling_coach_last.pt to resume, cycling_coach.pt for production.')


In [ ]:
# ── 6b. Fine-tune on GoldenCheetah (REAL data) — new two-family method ────────
# Keeps the synthetic "coach" (policy) intact while learning real FTP dose-
# response (forecast). Writes a SEPARATE checkpoint so you can gate before
# promoting — it never overwrites the good main model.
#
# To enable: right panel → Add data → add the GoldenCheetah OpenData dataset,
# then set FINETUNE = True.

FINETUNE         = False   # ← flip to True after adding the dataset
FORCE_FRESH_FT   = True    # ← True = delete old ft checkpoints and start clean
                           #   False = resume an interrupted fine-tune
GC_INPUT_DIR     = '/kaggle/input/goldencheetah-opendata-athlete-activity-and-mmp'
GC_PARQUET       = 'ml/data/goldencheetah.parquet'
FT_OUTPUT        = 'backend/models/cycling_coach_ft.pt'
FINETUNE_EPOCHS  = 20
FT_STEPS         = 2000
FINETUNE_LR      = 5e-5

if FINETUNE:
    import os, sys, argparse, subprocess, pandas as pd, torch, multiprocessing

    MODEL_FILE = globals().get('MODEL_FILE', 'backend/models/cycling_coach.pt')
    DATA_FILE  = globals().get('DATA_FILE',  'ml/data/synthetic.parquet')
    _ng = torch.cuda.device_count() if torch.cuda.is_available() else 1
    _vg = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
    _pg = 4096 if _vg >= 45 else 2048 if _vg >= 38 else 1024
    BATCH_SIZE = globals().get('BATCH_SIZE', _pg * _ng)
    os.environ.setdefault('DATALOADER_WORKERS', str(min(6, multiprocessing.cpu_count())))
    print(f'config: MODEL_FILE={MODEL_FILE}  batch={BATCH_SIZE}')

    assert os.path.exists(MODEL_FILE), 'Train/resume the main model first (cell 5).'

    # ── Optionally wipe old ft checkpoints for a clean start ─────────────────
    _ft_last = FT_OUTPUT.replace('.pt', '_last.pt')
    if FORCE_FRESH_FT:
        for _f in [FT_OUTPUT, _ft_last]:
            if os.path.exists(_f):
                os.remove(_f)
                print(f'Removed old checkpoint: {_f}')

    # ── Convert GoldenCheetah → our schema (stamps source=real) ──────────────
    if not os.path.exists(GC_PARQUET):
        if not os.path.isdir(GC_INPUT_DIR):
            raise FileNotFoundError(
                f'GoldenCheetah dataset not found at {GC_INPUT_DIR}. '
                'Add data → search "goldencheetah opendata".')
        print('Converting GoldenCheetah data…')
        env = os.environ.copy()
        env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"
        r = subprocess.run(
            [sys.executable, '-m', 'ml.training.convert_goldencheetah',
             '--input', GC_INPUT_DIR, '--output', GC_PARQUET],
            env=env, capture_output=False)
        if r.returncode != 0:
            raise RuntimeError('convert_goldencheetah failed — see output above')
    else:
        print(f'Using existing {GC_PARQUET}')

    gc_df = pd.read_parquet(GC_PARQUET)
    print(f'GoldenCheetah: {len(gc_df):,} rows, {gc_df["athlete_id"].nunique()} athletes')
    del gc_df
    assert os.path.exists(DATA_FILE), 'Synthetic parquet (for replay) missing — run cell 4.'

    for _k in list(sys.modules.keys()):
        if 'training.train' in _k:
            del sys.modules[_k]
    from ml.training.train import train as run_training

    # Resume or fresh start
    if not FORCE_FRESH_FT and os.path.exists(_ft_last):
        ft_checkpoint = _ft_last
        _reset_opt = False
        print(f'Resuming interrupted fine-tune from {_ft_last}')
    else:
        ft_checkpoint = MODEL_FILE
        _reset_opt = True
        print(f'Starting fresh fine-tune from {MODEL_FILE}')

    ft_args = argparse.Namespace(
        data              = f'{DATA_FILE}=synthetic,{GC_PARQUET}=real',
        output            = FT_OUTPUT,
        checkpoint        = ft_checkpoint,
        reset_optimizer   = _reset_opt,
        epochs            = FINETUNE_EPOCHS,
        batch_size        = BATCH_SIZE,
        steps_per_epoch   = FT_STEPS,
        lr                = FINETUNE_LR,
        seq_len           = 90,
        val_frac          = 0.15,
        seed              = 42,
        d_model           = 256,
        nhead             = 8,
        num_layers        = 8,
        d_ff              = 1024,
        dropout           = 0.1,
        fast              = False,
        patience          = 10,
        compile           = False,
        no_amp            = False,
        mask_workout_type = True,
        no_outcome_weighting      = False,
        outcome_scale             = 0.05,   # +5% FTP over 4w = full policy weight
        outcome_deadband          = 0.0,
        synthetic_forecast_weight = 0.0,    # synthetic FTP-delta is fiction → exclude
        forecast_min_delta        = 1e-4,   # real rows with ftp_delta≈0 → exclude
                                            # (athlete never tested — teaches "no effect")
    )
    run_training(ft_args)
    print(f'✓ Fine-tune complete -> {FT_OUTPUT}')
    print('Next: run the GATE cell to decide whether to promote it.')
else:
    print('Fine-tuning skipped (FINETUNE=False).')

In [ ]:
# ── 6c. GATE: did the fine-tune actually help? (run after fine-tune) ─────────
# Promote cycling_coach_ft.pt ONLY if its forecast calibration beats MAIN
# (lower FTPΔ_MAE, sign_acc/corr no worse) AND the coaching policy didn't
# collapse. compare_models prints the calibration table; sanity_check checks
# the policy. Self-contained so it works even after a kernel restart.
import os, subprocess, sys, glob

FT_OUTPUT  = globals().get('FT_OUTPUT',  'backend/models/cycling_coach_ft.pt')
MODEL_FILE = globals().get('MODEL_FILE', 'backend/models/cycling_coach.pt')
GC_PARQUET = globals().get('GC_PARQUET', '')

# Re-find the GC parquet if the var was lost on restart.
if not GC_PARQUET or not os.path.exists(GC_PARQUET):
    _cands = (glob.glob('/kaggle/input/**/goldencheetah*.parquet', recursive=True)
              + glob.glob('ml/data/goldencheetah.parquet'))
    GC_PARQUET = _cands[0] if _cands else GC_PARQUET

# If the BEST checkpoint is missing (interrupted run), fall back to _last.
if not os.path.exists(FT_OUTPUT):
    _ft_last = FT_OUTPUT.replace('.pt', '_last.pt')
    if os.path.exists(_ft_last):
        print('Best FT checkpoint missing — gating the interrupted run\'s last checkpoint:', _ft_last)
        FT_OUTPUT = _ft_last

env = os.environ.copy()
env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"

print('FT_OUTPUT :', FT_OUTPUT, '| exists:', os.path.exists(FT_OUTPUT))
print('GC_PARQUET:', GC_PARQUET, '| exists:', os.path.exists(GC_PARQUET))

if os.path.exists(FT_OUTPUT) and os.path.exists(GC_PARQUET):
    print('\n=== FORECAST CALIBRATION: MAIN vs FINE-TUNED ===')
    subprocess.run([sys.executable, 'ml/compare_models.py',
                    MODEL_FILE, FT_OUTPUT, GC_PARQUET], env=env)
    print('\n=== POLICY SANITY (fine-tuned must NOT collapse to recovery) ===')
    import shutil
    _bak = MODEL_FILE + '.gatebak'
    shutil.copy(MODEL_FILE, _bak)
    try:
        shutil.copy(FT_OUTPUT, MODEL_FILE)
        subprocess.run([sys.executable, 'ml/sanity_check.py'], env=env)
    finally:
        shutil.move(_bak, MODEL_FILE)   # restore main model untouched
    print('\nDecide: if calibration improved AND policy is sane -> promote FT to cycling_coach.pt')
else:
    print('\nMissing files — run the fine-tune cell (7) first, or fix the GC path above.')


In [ ]:
# ── 7. (Optional) Push model to GitHub ──────────────────────────────────────
# Create a PAT at https://github.com/settings/tokens (Classic, repo scope)
# then paste it when prompted.

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'kaggle@training'
!git config user.name  'Kaggle Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!cp /kaggle/working/cycling_coach.pt backend/models/cycling_coach.pt
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Kaggle GPU)"
!git push origin main
print('✓ Model pushed to GitHub!')

In [ ]:
# ── 8. Sanity check ──────────────────────────────────────────────────────────
import torch, sys
from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model=cfg.get('d_model', 128),
    nhead=cfg.get('nhead', 8),
    num_layers=cfg.get('num_layers', 6),
    dim_feedforward=cfg.get('dim_feedforward', 512),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()))
print('Best val loss:', ckpt.get('metrics', {}).get('val_loss', 'n/a'))
print('Epoch:',        ckpt.get('metrics', {}).get('epoch',    'n/a'))